In [1]:
import requests 
from bs4 import BeautifulSoup
import sqlite3
from urllib.parse import urljoin
import json

In [10]:
import requests
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0"
}

articles = []

for i in range(1,8):   
    url = f'https://www.mining.com/page/{i}?s=India'

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find all article cards on the search result page
        for article_tag in soup.find_all('div', class_='inner-content col-lg-8'):

            try:
                # Extract headline and link
                a_tag = article_tag.find('a')
                if not a_tag:
                    continue
                link = a_tag['href']
                headline = a_tag.get_text(strip=True)

                # Go to the article page to extract full content
                article_response = requests.get(link, headers=headers)
                article_soup = BeautifulSoup(article_response.text, 'html.parser')

                date_tag = article_soup.find('div', class_='post-meta mb-4')
                published_date = date_tag.get_text(strip=True) if date_tag else "N/A"

                content_tag = article_soup.find('div', class_='content')
                news = content_tag.get_text(strip=True) if content_tag else "N/A"

                articles.append({
                    "headline": headline,
                    "published_date": published_date,
                    "article_text": news
                })
            except Exception as inner_e:
                print(f"Error parsing individual article: {inner_e}")

    except Exception as e:
        print(f"Error fetching page {i}: {e}")

 


In [11]:
import pandas as pd
df = pd.DataFrame(articles)
df.head()

,headline,published_date,article_text
0,China pledges to address India’s rare earth needs,"Reuters| August 19, 2025 | 7:02 amCritical Min...",There is an upward trend in India-China relati...
1,"India’s IREL seeks Japan, South Korea partners...","Reuters| August 16, 2025 | 4:20 pmSuppliers & ...",India’s state-owned miner IREL is seeking to c...
2,Adani’s new copper smelter in India applies to...,"Reuters| August 13, 2025 | 10:03 amMarketsAsia...",A major new copper smelter in India owned by A...
3,India set to allow its private firms to mine a...,"Reuters| August 13, 2025 | 7:13 amEnergyAsiaUr...","India aims to allow private firms to mine, imp..."
4,India proposes minerals exchange to regulate m...,"Reuters| August 12, 2025 | 9:28 amMarketsAsiaI...",India’s Ministry of Mines has introduced a bil...


In [12]:
import re

def modify_date(text):
    match = re.search(r'[A-Za-z]+\s\d{1,2},\s\d{4}', text)

    if match:
        date = match.group()
        return date
    return None

In [13]:
df['published_date']=df['published_date'].apply(modify_date)

In [14]:
df.head()

,headline,published_date,article_text
0,China pledges to address India’s rare earth needs,"August 19, 2025",There is an upward trend in India-China relati...
1,"India’s IREL seeks Japan, South Korea partners...","August 16, 2025",India’s state-owned miner IREL is seeking to c...
2,Adani’s new copper smelter in India applies to...,"August 13, 2025",A major new copper smelter in India owned by A...
3,India set to allow its private firms to mine a...,"August 13, 2025","India aims to allow private firms to mine, imp..."
4,India proposes minerals exchange to regulate m...,"August 12, 2025",India’s Ministry of Mines has introduced a bil...


In [8]:
df['published_date']=pd.to_datetime(df['published_date'])

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   headline        40 non-null     object        
 1   published_date  40 non-null     datetime64[ns]
 2   article_text    40 non-null     object        
dtypes: datetime64[ns](1), object(2)
memory usage: 1.1+ KB


In [15]:
from transformers import pipeline

# Load the summarization pipeline with the BART model
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")


Device set to use cpu


In [13]:
def summarize_text(text):
    try:
         
        trimmed_text = text[:4000] if len(text) > 4000 else text

        summary = summarizer(trimmed_text, max_length=60, min_length=30, do_sample=False)
        return summary[0]['summary_text']
    except Exception as e:
        print(f"Error summarizing: {e}")
        return ""

In [15]:
df['summarized_text']=df['article_text'].apply(summarize_text)

In [16]:
df.head()

,headline,published_date,article_text,summarized_text
0,High prices stifle gold demand in top Asian hu...,2025-07-25,Physical gold demand in key Asian hubs was sub...,Physical gold demand in key Asian hubs was sub...
1,"India in talks with Chile, Peru to source crit...",2025-07-15,India is holding talks with Chile and Peru to ...,India is holding talks with Chile and Peru to ...
2,Share of Russian aluminum in LME warehouses fa...,2025-07-10,The share of available aluminum stocks of Russ...,The share of available aluminum stocks of Russ...
3,India plans to kickstart rare earth output to ...,2025-07-09,A proposed plan by India to spur local product...,India is planning an incentive program worth a...
4,"India to woo foreign copper miners, expand ties",2025-07-04,India on Friday unveiled a series of steps to ...,India may have to import 91%-97% of its copper...


In [17]:
df.to_csv('news_articles.csv')

In [ ]:
def generate_collapsible_news_html(df, output_file="mining_news_collapsible.html"):
    html_start = """
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Mining News Summary</title>
        <link rel="stylesheet" href="{{ url_for('static', filename='news.css') }}">
        <style>
            body {
                font-family: Arial, sans-serif;
            }
            .news-container {
                max-width: 900px;
                margin: 30px auto;
                padding: 20px;
            }
            .news-item {
                border: 1px solid #ccc;
                border-radius: 8px;
                margin-bottom: 20px;
                padding: 15px;
                background-color: #f9f9f9;
            }
            .news-header h1 {
                text-align: center;
                margin-bottom: 30px;
            }
            .news-date {
                color: #777;
                font-size: 0.9em;
                margin-bottom: 10px;
            }
            .read-more {
                color: #007BFF;
                cursor: pointer;
                font-weight: bold;
            }
            .summary-text {
                display: none;
                margin-top: 10px;
            }
        </style>
        <script>
            function toggleSummary(id) {
                const element = document.getElementById(id);
                if (element.style.display === "none") {
                    element.style.display = "block";
                } else {
                    element.style.display = "none";
                }
            }
        </script>
    </head>
    <body>
        <div class="news-container">
            <div class="news-header">
                <h1>📰 Mining News Summary</h1>
            </div>
    """

    html_end = """
        </div>
    </body>
    </html>
    """

    news_items = ""
    for idx, row in df.iterrows():
        summary_id = f"summary_{idx}"
        news_items += f"""
        <div class="news-item">
            <h3>{row['headline']}</h3>
            <p class="news-date">{row['published_date']}</p>
            <p class="read-more" onclick="toggleSummary('{summary_id}')">🔽 Read More</p>
            <p id="{summary_id}" class="summary-text"><strong>Summary:</strong> {row['summarized_text']}</p>
        </div>
        """

    full_html = html_start + news_items + html_end

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(full_html)

    print(f"✅ Collapsible news HTML saved to {output_file}")


In [ ]:
def generate_news_html(df, output_file="mining_news_summary.html"):
    html_start = """
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Mining News Summary</title>
        <link rel="stylesheet" href="{{ url_for('static', filename='news.css') }}">
    </head>
    <body>
        <div class="news-container">
            <div class="news-header">
                <h1>📰 Mining News Summary</h1>
                <a href="/" class="back-btn">← Back to Chat</a>
            </div>
            <div class="news-content">
    """

    html_end = """
            </div>
        </div>
    </body>
    </html>
    """

    # Build news items
    news_items = ""
    for _, row in df.iterrows():
        news_items += f"""
        <div class="news-item">
            <div class="news-text">
                <h3>{row['headline']}</h3>
                <p class="news-date">{row['published_date']}</p>
                <p><strong>Original:</strong> {row['article_text']}</p>
                <p><strong>Summary:</strong> {row['summarized_text']}</p>
            </div>
        </div>
        """

    # Combine and write to file
    full_html = html_start + news_items + html_end

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(full_html)

    print(f"✅ News summary saved to {output_file}")


In [ ]:
generate_collapsible_news_html(df)


✅ Collapsible news HTML saved to mining_news_collapsible.html
